# Conformal Benchmark

This notebook runs conformal risk control ([conformal-protein-retrieval](https://github.com/ronboger/conformal-protein-retrieval)), the baseline PLM-Caliper is compared with for controlling false discoveries in protein retrieval. Conformal risk control learns one score cutoff $\hat{\lambda}$ per target FDR level from labelled calibration queries and applies it to held-out test queries. It needs only the search scores of the ASTRAL40 all-vs-all search, with SCOP homology as the ground truth, so PLM-Caliper is not run here.

## 0. Setup

Load the libraries and the conformal risk control code, then set the paths and the search method.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from Bio import SeqIO

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, f"{REPO_ROOT}/libs/conformal-protein-retrieval")
from protein_conformal import util as conformal  # conformal risk control code, imported unmodified

In [ ]:
searching_method = "dhr" 
search2score_mapping = {"blastp": "blastp_postprocessed", "plm": "plm", "tmvec": "tmvec", "dhr": "dhr_postprocess"}
score_method = search2score_mapping[searching_method]
SEARCH_LABEL = {"blastp": "BLASTp", "plm": "PLMsearch", "tmvec": "TMvec", "dhr": "DHR"}

QUERY_NAME = "astral" 
TARGET_NAME = "astral"
Q_LEVELS = [round(0.05 * i, 2) for i in range(1, 20)]  # target FDR levels 0.05, 0.10, ..., 0.95

DATA_DIR = f"{REPO_ROOT}/data"
OUT_DIR = f"{DATA_DIR}/conformal"
CACHE_DIR = f"{OUT_DIR}/cache"
FIG_DIR = f"{REPO_ROOT}/results/conformal"

## 1. Searching score

Read the real query scores into two dense query × target matrices: the search scores, and the SCOP homology level of every pair (-1 different fold, 0 same fold, 1 same superfamily, 2 same family). Rows and columns follow one id list, so entry (i, j) is query i against target j. The matrices are cached in `data/conformal/cache/`.

**Note:** The score table is the real query table from step 2 of the ablation notebook, also on [Query Searching Scores](https://huggingface.co/datasets/yifanyang993/PLMCaliper/tree/main/raw_searching_data). Put it here (DHR as an example):

```text
data/
└── result_dhr_postprocess_astral_astral.txt   # real query scores
```

In [ ]:
src = f"{DATA_DIR}/result_{score_method}_{QUERY_NAME}_{TARGET_NAME}.txt"
cache = f"{CACHE_DIR}/{score_method}"
if all(os.path.isfile(f"{cache}_{f}") for f in ("sims.npy", "homo.npy", "ids.txt")):
    print(f"[skip] For time-saving, the {score_method} matrices already exist in {CACHE_DIR}")
else:
    # every query is scored against all targets, so the first query's rows list the target ids
    head = pd.read_csv(src, sep="\t", usecols=["qid", "tid"], nrows=1_000_000)
    n = int((head["qid"] == head["qid"].iloc[0]).sum())
    ids = pd.Index(head["tid"].iloc[:n])

    # fill the matrices 200 queries at a time
    sims = np.zeros((n, n), dtype=np.float32)
    homo = np.zeros((n, n), dtype=np.int8)
    for chunk in pd.read_csv(src, sep="\t", usecols=["qid", "tid", "score", "homo_type"], chunksize=n * 200,
                             dtype={"score": np.float32, "homo_type": np.int8}):
        rows = ids.get_indexer(chunk["qid"].iloc[::n])[:, None]
        cols = ids.get_indexer(chunk["tid"]).reshape(-1, n)
        sims[rows, cols] = chunk["score"].to_numpy().reshape(-1, n)
        homo[rows, cols] = chunk["homo_type"].to_numpy().reshape(-1, n)

    os.makedirs(CACHE_DIR, exist_ok=True)
    np.save(f"{cache}_sims.npy", sims)
    np.save(f"{cache}_homo.npy", homo)
    Path(f"{cache}_ids.txt").write_text("\n".join(ids) + "\n")
    print(f"[OK] {n} x {n} score and homology matrices -> {CACHE_DIR}")

Rescale the scores to [0, 1], the range the conformal method searches $\lambda$ in. A target is a true homolog when it shares the query's superfamily or family. Same-fold pairs from different superfamilies are uncertain, so they are excluded by setting their score to -1e-6, below every real score.

In [ ]:
sims = np.load(f"{cache}_sims.npy")
homo = np.load(f"{cache}_homo.npy")
ids = Path(f"{cache}_ids.txt").read_text().split()

lo, hi = float(sims.min()), float(sims.max())
sims -= lo
sims /= (hi - lo)
labels = np.isin(homo, (1, 2)).astype(np.int8)
sims[homo == 0] = -1e-6

## 2. Split

`make_split` draws `n_trials` calibration/test splits. Queries are grouped by `split_level` and whole groups go to one side: with `fold`, no test query has a calibration query from its SCOP fold, while `random` splits single queries, as in the conformal paper. Groups are drawn in random order until about `n_calib` queries are calibration queries, skipping any group that would overshoot `n_calib` by more than 50%.

In [ ]:
def make_split(ids, split_level, n_calib=1000, n_trials=5, seed=0):
    if split_level == "random":  # every query is its own group
        groups = np.arange(len(ids))
    else:  # queries sharing a SCOP fold, superfamily or family form one group
        depth = {"fold": 2, "superfamily": 3, "family": 4}[split_level]
        sccs = {rec.id: rec.description.split()[1] for rec in SeqIO.parse(f"{DATA_DIR}/{QUERY_NAME}.fa", "fasta")}
        groups = np.unique([".".join(sccs[i].split(".")[:depth]) for i in ids], return_inverse=True)[1]
    members = [np.flatnonzero(groups == g) for g in range(groups.max() + 1)]

    rng = np.random.default_rng(seed)
    splits = []
    for _ in range(n_trials):
        cal, n_cal = [], 0
        for g in rng.permutation(len(members)):
            if n_cal >= n_calib:
                break
            if cal and n_cal + len(members[g]) > n_calib * 1.5:
                continue
            cal.append(members[g])
            n_cal += len(members[g])
        cal = np.concatenate(cal)
        splits.append((cal, np.setdiff1d(np.arange(len(ids)), cal)))
    return splits

## 3. Conformal

`conformal_threshold` learns the cutoffs from the calibration queries of each split. It computes the false discovery proportion (FDP) of every calibration query on a grid of `n_lambda` score cutoffs. For each target FDR level q, Learn then Test (LTT) picks $\hat{\lambda}$, the smallest cutoff that, together with every larger cutoff, passes the CLT test of mean FDP ≤ q at p-value `delta`. The cutoffs are saved to `data/conformal/` (DHR with fold splits as an example):

```text
data/conformal/
└── conformal_threshold_dhr_postprocess_astral_fold.csv   # one cutoff per split and target FDR level
```

In [ ]:
def conformal_threshold(sims, labels, splits, out, delta=0.5, n_lambda=5000):
    rows = []
    for trial, (cal, _) in enumerate(splits):
        # FDP of every calibration query at every cutoff on the grid
        lambdas = np.linspace(sims[cal].min(), sims[cal].max(), n_lambda)
        cal_fdp = np.empty((len(cal), n_lambda))
        for r, i in enumerate(cal):
            homologs = np.sort(sims[i][labels[i] == 1])
            n_hits = sims.shape[1] - np.searchsorted(np.sort(sims[i]), lambdas)
            n_homo = len(homologs) - np.searchsorted(homologs, lambdas)
            cal_fdp[r] = (n_hits - n_homo) / np.maximum(n_hits, 1)
        risks, stds = cal_fdp.mean(axis=0), np.maximum(cal_fdp.std(axis=0), 1e-6)

        # LTT: the smallest cutoff that passes, together with every larger cutoff
        for q in Q_LEVELS:
            passed = np.cumprod((conformal.clt_p_value(risks, stds, len(cal), q) <= delta)[::-1])[::-1]
            rows.append({"trial": trial, "q": q, "lambda": min(float(lambdas[np.argmax(passed)]), 1.0)})
    thresholds = pd.DataFrame(rows)
    thresholds.to_csv(out, index=False)
    return thresholds

## 4. Plot

`plot_fdr` and `plot_power` apply the cutoffs of each split to its test queries, and plot the real FDR and power at each target FDR level, averaged over the test queries, with the 95% CI across them. As in the conformal paper, a query's FDP is the share of its selected targets that are not homologs, and 0 when it selects nothing. The figures go to `results/conformal/`.

The last cell runs split + conformal + plot for fold and random splits.

In [ ]:
# set plotting pattern
matplotlib.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "Nimbus Sans", "Liberation Sans", "TeX Gyre Heros", "DejaVu Sans"]
                       + list(matplotlib.rcParams["font.sans-serif"]),
    "axes.unicode_minus": False, "font.size": 6, "axes.labelsize": 7, "axes.titlesize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "axes.linewidth": 0.6, "lines.linewidth": 0.9, "lines.markersize": 3.0, "grid.linewidth": 0.4,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "savefig.dpi": 300, "figure.dpi": 150, "legend.frameon": True, "legend.framealpha": 1.0,
})
CMAP = mcolors.LinearSegmentedColormap.from_list(
    "fdr_mako", ["#0b0405", "#28192e", "#3b2f5f", "#3f4a8f", "#366ca0", "#348da7", "#39acac", "#59ccad", "#a9e1bd"])
X = np.arange(len(Q_LEVELS))
os.makedirs(FIG_DIR, exist_ok=True)

# define utils for plotting
def bar_panel(values, ci):
    # one bar per target FDR level, with its 95% CI and its value written above
    fig_w, fig_h = 5.92, 5.25
    fig = plt.figure(figsize=(fig_w, fig_h))
    ax = fig.add_axes([0.6 / fig_w, 0.6 / fig_h, 4.3 / fig_w, 4.3 / fig_h])
    ax.set_box_aspect(1)
    ax.bar(X, values, width=0.82, color=[CMAP(i / (len(X) - 1)) for i in X], edgecolor="none", zorder=2)
    ax.errorbar(X, values, yerr=ci, fmt="none", ecolor="black", elinewidth=0.9, capsize=0, zorder=4, label="95% CI")
    gap = 0.015 * max(float(np.max(values + ci)), 1.0)
    for xi, v, t in zip(X, values, values + ci):
        ax.text(xi, t + gap, f"{v:.2f}", ha="center", va="bottom", fontsize=5, fontweight="bold", zorder=5)
    ax.set_xlim(-0.6, len(X) - 0.4)
    ax.set_xticks(X)
    ax.set_xticklabels([f"{q:.2f}" for q in Q_LEVELS], rotation=45, ha="right")
    ax.set_axisbelow(True)
    return fig, ax

def finish_plot(fig, ax, ylabel, title, out):
    ax.set_xlabel("Target FDR Level", fontsize=7, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=7, fontweight="bold")
    ax.set_title(title, fontsize=7, fontweight="bold")
    # put the legend right of the plot, and widen the figure to fit it
    pos = ax.get_position()
    legend = ax.legend(fontsize=6, loc="upper left", bbox_to_anchor=(pos.x1 + 0.02, pos.y1),
                       bbox_transform=fig.transFigure, frameon=True, fancybox=False, edgecolor="0.35",
                       facecolor="white", borderaxespad=0.0)
    fig.canvas.draw()
    pos = ax.get_position()
    w, h = fig.get_size_inches()
    left, bottom, width, height = pos.x0 * w, pos.y0 * h, pos.width * w, pos.height * h
    new_w = max(left + width, legend.get_window_extent(fig.canvas.get_renderer()).x1 / fig.dpi) + 0.18
    if new_w > w + 1e-6:
        fig.set_size_inches(new_w, h, forward=True)
        ax.set_position([left / new_w, bottom / h, width / new_w, height / h])
        legend.set_bbox_to_anchor(((left + width + 0.12) / new_w, ax.get_position().y1), transform=fig.transFigure)
    fig.savefig(out)
    plt.show()

In [ ]:
def plot_fdr(sims, labels, splits, thresholds, split_level):
    for trial, (_, test) in enumerate(splits):
        sims_test, labels_test = sims[test], labels[test]
        # FDP of every test query at each cutoff
        fdp = []
        for lhat in thresholds.loc[thresholds["trial"] == trial, "lambda"]:
            sel = sims_test >= lhat
            n_sel = sel.sum(axis=1)
            fdp.append((n_sel - (sel & (labels_test == 1)).sum(axis=1)) / np.maximum(n_sel, 1))
        fdp = np.array(fdp)
        values = fdp.mean(axis=1)
        ci = 1.96 * fdp.std(axis=1, ddof=1) / np.sqrt(fdp.shape[1])

        fig, ax = bar_panel(values, ci)
        ax.plot(X, Q_LEVELS, linestyle="--", color="gray", linewidth=0.9, marker="o", markersize=1.8, zorder=3,
                label="Target FDR")
        ax.set_ylim(0, max(float(np.max(values + ci)) * 1.12, 1.05))
        finish_plot(fig, ax, "Real FDR", f"FDR Control ({SEARCH_LABEL[searching_method]}, Conformal, trial {trial + 1})",
                    f"{FIG_DIR}/{score_method}_{split_level}_trial{trial + 1}_conformal_FDR.pdf")

def plot_power(sims, labels, splits, thresholds, split_level):
    for trial, (_, test) in enumerate(splits):
        sims_test, labels_test = sims[test], labels[test]
        # power of every test query at each cutoff
        power = []
        for lhat in thresholds.loc[thresholds["trial"] == trial, "lambda"]:
            power.append(((sims_test >= lhat) & (labels_test == 1)).sum(axis=1) / np.maximum(labels_test.sum(axis=1), 1))
        power = np.array(power)
        values = power.mean(axis=1)
        ci = 1.96 * power.std(axis=1, ddof=1) / np.sqrt(power.shape[1])

        fig, ax = bar_panel(values, ci)
        ax.set_ylim(0, 1.12)
        finish_plot(fig, ax, "Average Power", f"Power ({SEARCH_LABEL[searching_method]}, Conformal, trial {trial + 1})",
                    f"{FIG_DIR}/{score_method}_{split_level}_trial{trial + 1}_conformal_Power.pdf")

In [ ]:
# split + conformal + plot, for fold and random splits
for split_level in ("fold", "random"):
    print(f"\n>>> {split_level} splits")
    splits = make_split(ids, split_level, n_calib=1000, n_trials=5, seed=0)
    thresholds = conformal_threshold(sims, labels, splits, f"{OUT_DIR}/conformal_threshold_{score_method}_{QUERY_NAME}_{split_level}.csv",
                                     delta=0.5, n_lambda=5000)
    plot_fdr(sims, labels, splits, thresholds, split_level)
    plot_power(sims, labels, splits, thresholds, split_level)